# ICORE — choose one problem or compare all 393

This notebook uses **arithmetic mean time in milliseconds**, never silently drops failed solves,
and shows faster, slower, tied, and incomparable cases separately.

To learn how to build your own model, start with [Tutorial 1](tutorials/01_scalar_regulator.ipynb).
All stored coefficient examples are in `problems/`. The default comparison methods are
ICORE C, ICORE C++, HPIPM BALANCE, and QOCO.

> **Development transparency:** Substantial portions of ICORE's implementation and documentation were generated and revised with GPT under human direction. Not all details have undergone independent human review. See the [development and review statement](README.md#development-transparency-and-review-status).


In [1]:
# Use a fresh kernel: limit threaded BLAS before importing NumPy/pandas.
import os
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[name] = "1"

from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "icore" / "api.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from example_tools.catalog import list_problems, load_problem
from example_tools.runner import prepare, run_problem, run_all, mean_table
from example_tools.display import show_catalog, show_problem, show_decisions, show_comparisons, export_tables
from example_tools.trace import replay

METHODS = ("ICORE C", "ICORE C++", "HPIPM", "QOCO")
TOLERANCE = 1e-8
N = 5                          # Number of cold solves per method, not IPM iterations.
prepared = prepare(METHODS)     # Native compilation is outside measured solve time.
display(pd.DataFrame(prepared["solvers"]).drop(columns=["compiler"], errors="ignore"))

,solver,status,version,target,precision
0,ICORE C,ready,0.20.0,GENERIC,float64; optional guarded mixed OCP kernels
1,ICORE C++,ready,0.20.0,GENERIC,float64; optional guarded mixed OCP kernels
2,HPIPM,ready,HPIPM,BLASFEO,float64
3,QOCO,ready,0.3.2,sparse LDL,float64


## 1. Browse the catalog
Filter by a name fragment or family; these filters only change the display.
393 coefficient instances include synthetic tests and model updates, not 393 independent applications.

In [2]:
show_catalog()
FILTER = ""                 # Set to "" to show every problem.
entries = list_problems(contains=FILTER)
display(pd.DataFrame(entries)[["name", "family", "N", "nx", "nu", "n_inequalities"]])

### 393 distinct coefficient instances

Group,Problem
Main benchmark set,175
Additional validation models,72
Distinct matrix-update states,32
Analytic KKT examples,114


Repeated backend runs and aliases of identical coefficients are not counted as new problems. Matrix updates, analytic tests, and equivalent formulations are labeled separately; this is not a count of independent application families.

,name,family,N,nx,nu,n_inequalities
0,C01_dense_qp,dense_qp,35,6,3.0,576
1,C02_dense_qp,dense_qp,50,10,4.0,969
2,C03_dense_qp,dense_qp,70,12,5.0,1562
3,C04_dense_qp,dense_qp,90,18,7.0,2639
4,C05_dense_qp,dense_qp,120,24,10.0,4598
...,...,...,...,...,...,...
388,KKT_n033_mixed_offset3,analytic_mixed,1,1,33.0,47
389,KKT_n033_mixed_offset7,analytic_mixed,1,1,33.0,47
390,KKT_n033_mixed_holes,analytic_mixed,1,1,33.0,47
391,KKT_n033_mixed_permuted,analytic_mixed,1,1,33.0,47


## 2. Select, display, and solve one problem

In [3]:
PROBLEM_NAME = "QUADCMPC1"
problem = load_problem(PROBLEM_NAME)
formulation = show_problem(problem)
selected = run_problem(problem, repetitions=N, methods=METHODS,
                       tolerance=TOLERANCE, prepared=prepared)
display(mean_table(selected["rows"], repetitions=N))

## Problem being solved: `QUADCMPC1`
**Stage-structured quadratic OCP**

<IPython.core.display.Math object>

<IPython.core.display.Math object>

The symbols are the selected input blocks Q/R/S/q/r/A/B/b/G/h/x0/F/f; at stages without SOCs the cone condition reduces to G_k z_k ≤ h_k. For QOCO the same model is assembled in sparse conic form, x₀ is substituted, and results are restored to original coordinates.

Symbol / property,Value
N,32
n_x,12
n_u,6
Scalar inequality/conic rows,396
Equalities,384
Terminal F rows,0
SOC blocks,none
Orthant rows (per stage),12


**Cohort:** Main benchmark set. A reference objective value is available.

,problem,solver,records,passed,expected,measured,eligible,setup_mean_ms,solve_mean_ms,total_mean_ms,compile_mean_ms,solve_std_ms,outcomes,error
0,QUADCMPC1,HPIPM,5,5,5,5,True,53.944133,3.547600,57.491733,0.0,0.539090,passed:5,
1,QUADCMPC1,ICORE C++,5,5,5,5,True,45.523200,1.366709,46.889909,0.0,0.141069,passed:5,
2,QUADCMPC1,ICORE C,5,5,5,5,True,44.128192,1.399025,45.527217,0.0,0.209268,passed:5,
3,QUADCMPC1,QOCO,5,5,5,5,True,42.365150,3.073825,45.438975,0.0,0.818165,passed:5,


## 3. Inspect the actual execution
Basic counters come from the measured production solve. Set `DETAILED=True` for one extra,
untimed diagnostic solve. Its output must match production before its trace is used.

In [4]:
BACKEND = "cpp"
DETAILED = False
method = "ICORE C++" if BACKEND == "cpp" else "ICORE C"
trace = None
if DETAILED:
    trace = replay(problem, backend=BACKEND, tolerance=TOLERANCE,
                   production_result=selected["latest_solutions"].get(method))
show_decisions(problem, selected, backend=BACKEND, trace=trace)

### Observed execution route

Question,Answer,Evidence
Which kernel was selected?,General OCP / Riccati kernel,Production solve #5: last_kernel
Which structural property justified it?,General stage-structured OCP route. Executed rejection checkpoints for specialized routes appear in the diagnostic replay below.,Input + native certificate/kernel counter
Which factorization was attempted?,Production counters are shown below; they do not record every attempt in order. Enable diagnostic replay for details.,A plan is not treated as an executed operation
Which fallback actually occurred?,dense_fallbacks=0; factor_retries=0; refinements=0,Counters from the timed production solve
Which arithmetic precision was used?,Outer variables/residuals use FP64; the production build does not expose every internal mixed-precision event.,"Interface dtype + observed acceptance events, when available"
Why was a fast path rejected?,Standard counters do not contain this detail; diagnostic replay is disabled.,Separate replay; no inferred rejection reason


Native counter,Value
last_kernel,0
dense_face_attempts,0
dense_face_steps,0
dense_face_accepted,0
dense_fallbacks,0
dense_packed,0
ocp_packed_stages,0
ocp_packed_rejected,0
ocp_sqrt_factors,0
ocp_sqrt_rejected,0


## 4. All mode
Set `RUN_ALL=True` to run **every catalog problem** with each method `ALL_N` times.
Each repetition uses a fresh workspace and a cold solve. Compilation, model conversion,
correctness audits, and reporting are excluded from solve timing.

Results are saved after each problem. `RESUME=True` resumes only a compatible saved run.
Setup and solve means are reported separately; `total_ms` means setup + solve, not the entire application.
The detailed diagnostic replay above is never part of All mode.

In [5]:
RUN_ALL = True
ALL_N = 5
OUTPUT = ROOT / "runs" / f"all393_N{ALL_N}"
RESUME = False
all_results = None

def progress(state):
    if state["completed"] % 25 == 0 or state["complete"]:
        print(f"{state['completed']}/{state['total']} saved; last: {state['last']}")

if RUN_ALL:
    all_results = run_all(repetitions=ALL_N, methods=METHODS, tolerance=TOLERANCE,
                          prepared=prepared, output_dir=OUTPUT, resume=RESUME, progress=progress)
    print("Completed:", all_results["complete"])
else:
    print("Set RUN_ALL=True to run all 393 problems.")

25/393 saved; last: C25_dense_soc
50/393 saved; last: C50_l1_epigraph
75/393 saved; last: QUADCMPC3
100/393 saved; last: LIPMWALK23
125/393 saved; last: WHLIPBAL18
150/393 saved; last: OSQP_QUADCOPTER_N030
175/393 saved; last: SHQ_O12_varying_N15_x16_u05
200/393 saved; last: holdout_chain_00_original
225/393 saved; last: validation_other24_dense_00_equivalent
250/393 saved; last: OSQP_CONTROL_n040_seed1__matrix_03
275/393 saved; last: HPIPM_MASS_SPRING_N030__matrix_04
300/393 saved; last: KKT_n003_boxes_offset1
325/393 saved; last: KKT_n004_general_offset1
350/393 saved; last: KKT_n007_mixed_offset1
375/393 saved; last: KKT_n033_boxes_prefix
393/393 saved; last: KKT_n033_empty_prefix
Completed: True


## 5. Faster and slower tables
By default the reference is the **fastest valid requested rival** for each problem.
Use `RIVALS=("HPIPM",)` or `("QOCO",)` to compare with one rival.
A method must pass all N/N accuracy checks; failures and unsupported cases do not become speed wins.
Partial rival coverage is explicitly labeled. A faster mean is not a statistical significance claim.

In [6]:
ICORE_METHOD = "ICORE C++"      # Or "ICORE C".
RIVALS = ("HPIPM", "QOCO")
METRIC = "solve_ms"             # Or "setup_ms", "total_ms".

results_to_show = all_results if all_results is not None else selected
N_to_show = results_to_show["repetitions"]
tables = show_comparisons(results_to_show["rows"], repetitions=N_to_show,
                          icore_solver=ICORE_METHOD, rivals=RIVALS, metric=METRIC)
if all_results is not None:
    export_tables(tables, OUTPUT / "tables")

## All mode: ICORE C++ / HPIPM, QOCO
**N=5; metric=solve_ms; arithmetic mean; time unit ms.** Each problem is compared with the fastest requested rival that passes N/N accuracy checks. A row with `coverage=partial` is not a win over an unavailable or unsuccessful rival. These are descriptive measurements, not significance tests or claims of superiority on all hardware.

### ICORE faster — 292 problems

problem,ICORE solver,HPIPM mean ms,HPIPM pass,ICORE C++ mean ms,ICORE C++ pass,ICORE C mean ms,ICORE C pass,QOCO mean ms,QOCO pass,valid rivals,coverage,fastest valid rival,ICORE / rival,ICORE change %
C01_dense_qp,ICORE C++,0.686083,5/5,0.518533,5/5,0.539433,5/5,1.418250,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,0.755788,-24.421231
C06_sparse_qp,ICORE C++,0.744891,5/5,0.549533,5/5,0.536800,5/5,1.245050,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,0.737736,-26.226398
C07_sparse_qp,ICORE C++,1.757467,5/5,1.429000,5/5,1.454617,5/5,3.493041,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,0.813102,-18.689785
C08_sparse_qp,ICORE C++,3.632459,5/5,2.380142,5/5,2.513692,5/5,8.282700,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,0.655243,-34.475741
C09_sparse_qp,ICORE C++,9.504292,5/5,5.378017,5/5,5.564992,5/5,22.601767,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,0.565851,-43.414862
C10_sparse_qp,ICORE C++,24.843858,5/5,15.709808,5/5,15.476708,5/5,63.152350,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,0.632342,-36.765827
C11_ill_conditioned_qp,ICORE C++,0.633033,5/5,0.497766,5/5,0.494308,5/5,1.184942,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,0.786319,-21.368062
C12_ill_conditioned_qp,ICORE C++,1.534650,5/5,1.515517,5/5,1.554000,5/5,3.784534,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,0.987532,-1.246758
C13_ill_conditioned_qp,ICORE C++,2.744950,5/5,2.665825,5/5,2.675917,5/5,7.981775,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,0.971174,-2.882566
C15_ill_conditioned_qp,ICORE C++,19.790300,5/5,16.712375,5/5,17.446600,5/5,71.215041,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,0.844473,-15.552696


### ICORE slower — 68 problems

problem,ICORE solver,HPIPM mean ms,HPIPM pass,ICORE C++ mean ms,ICORE C++ pass,QOCO mean ms,QOCO pass,ICORE C mean ms,ICORE C pass,valid rivals,coverage,fastest valid rival,ICORE / rival,ICORE change %
C02_dense_qp,ICORE C++,1.410116,5/5,1.500025,5/5,4.432083,5/5,1.516883,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,1.063760,6.375974
C03_dense_qp,ICORE C++,3.191809,5/5,3.397109,5/5,9.406691,5/5,3.399825,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,1.064321,6.432091
C04_dense_qp,ICORE C++,7.098258,5/5,8.253108,5/5,27.757267,5/5,8.405783,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,1.162695,16.269487
C05_dense_qp,ICORE C++,17.647033,5/5,18.520108,5/5,78.368217,5/5,18.959634,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,1.049474,4.947432
C14_ill_conditioned_qp,ICORE C++,7.087608,5/5,8.363592,5/5,28.369125,5/5,8.604250,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,1.180030,18.003021
C19_long_dynamics,ICORE C++,11.176250,5/5,19.800142,5/5,42.271250,5/5,19.040233,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,1.771627,77.162662
rocket_first_qp,ICORE C++,27.759342,5/5,2.783566,5/5,2.231583,5/5,2.717325,5/5,"HPIPM, QOCO",all requested rivals valid,QOCO,1.247350,24.735049
historical_qp_easy_N010,ICORE C++,0.087633,5/5,0.089150,5/5,0.148192,5/5,0.086075,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,1.017304,1.730367
historical_qp_easy_N080,ICORE C++,0.715575,5/5,0.931325,5/5,1.452583,5/5,0.863150,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,1.301506,30.150575
historical_qp_masses_N010,ICORE C++,0.260225,5/5,0.327833,5/5,0.790525,5/5,0.335925,5/5,"HPIPM, QOCO",all requested rivals valid,HPIPM,1.259808,25.980750


### Tied within the selected threshold — 0 problems

No problems in this group.

### Incomparable / failed accuracy gate — 33 problems

problem,ICORE solver,QOCO mean ms,QOCO pass,ICORE C mean ms,ICORE C pass,HPIPM mean ms,HPIPM pass,ICORE C++ mean ms,ICORE C++ pass,valid rivals,coverage,reason
C21_dense_soc,ICORE C++,5.734717,0/5,1.854642,5/5,—,0/5,1.807909,5/5,,partial / unavailable / failed rival,No valid rival
C22_dense_soc,ICORE C++,15.856025,0/5,4.218317,5/5,—,0/5,3.982342,5/5,,partial / unavailable / failed rival,No valid rival
C23_dense_soc,ICORE C++,31.938275,0/5,8.219925,5/5,—,0/5,8.220400,5/5,,partial / unavailable / failed rival,No valid rival
C24_dense_soc,ICORE C++,91.296358,0/5,22.422542,5/5,—,0/5,17.910525,5/5,,partial / unavailable / failed rival,No valid rival
C25_dense_soc,ICORE C++,321.932192,0/5,37.567292,5/5,—,0/5,37.149617,5/5,,partial / unavailable / failed rival,No valid rival
C26_local_soc,ICORE C++,3.940692,0/5,1.479758,5/5,—,0/5,1.468725,5/5,,partial / unavailable / failed rival,No valid rival
C27_local_soc,ICORE C++,13.116950,0/5,3.717675,5/5,—,0/5,3.619675,5/5,,partial / unavailable / failed rival,No valid rival
C28_local_soc,ICORE C++,23.728792,0/5,7.032467,5/5,—,0/5,7.051300,5/5,,partial / unavailable / failed rival,No valid rival
C29_local_soc,ICORE C++,45.930917,0/5,12.676150,5/5,—,0/5,12.626425,5/5,,partial / unavailable / failed rival,No valid rival
C30_local_soc,ICORE C++,106.927992,0/5,26.795467,5/5,—,0/5,26.175767,5/5,,partial / unavailable / failed rival,No valid rival


## Notes
- All numeric tolerances and models remain unchanged by this reporting layer.
- The HPIPM QP adapter explicitly skips generic SOCs; the model is not approximated.
- Model assembly, first compilation, setup, solver time, and complete application latency are different quantities.
- Use larger N and repeated sessions for performance decisions. Hardware and CPU load matter.
- Runs and build caches are ignored by Git; they are not part of the source repository.
- See [benchmark details](docs/BENCHMARKING.md) and [the API contract](docs/API.md).